# 07 · Choose a model, check its answer

**General question → Switchyard → Lightning base**

**Classification → Switchyard → Lightning specialist**

These routes are selected explicitly by the tool. Switchyard resolves the named route; it does not infer the task from the question in this example.

**Who does what:** Nano orchestrates the lesson tools. Switchyard routes their requests to Lightning base or specialist. Application code validates the returned classification.

###  explicit routing and application contracts

the [Switchyard TOML reference](https://nvidia-nemo.github.io/Switchyard/reference/toml_schema/) explains the routing configuration. The workshop pins `nemo-switchyard==0.3.0`; the current guide may show newer interfaces. Our exact configuration is [switchyard.toml](../run/workshop/configs/switchyard.toml).

start the Lightning endpoint with both aliases, then run the route, gate, and ordering checks. The cells below write an editable TOML file and open an embedded `Server`;  you do not start a separate Switchyard daemon.

**What is enforced:** `workshop-reason` targets `base`; `workshop-triage` targets `specialist`. [contracts.py](../run/workshop/workshop_agent/contracts.py) checks labels/confidence and assigns the destination by lookup. [runtime.py](agent/runtime.py) blocks drafting before evidence and valid classification.

these are fixed routes and deterministic gates. Switchyard does not learn model selection here, and a corrected destination is application behavior. Inspect the `validate_item` and `require_draft_inputs` cells to distinguish model output from code-enforced decisions.

**Exact routing reference:** NVIDIA’s [TOML schema](https://nvidia-nemo.github.io/Switchyard/reference/toml_schema/) documents `llm_clients`, `targets`, `routes`, and `passthrough`; clients send the route’s `id` as the model name. Compare it with [our configuration](../run/workshop/configs/switchyard.toml) and [embedded Python wrapper](agent/routing_trace.py), pinned by `nemo-switchyard==0.3.0`. The current quickstart teaches a development Rust API and is conceptual background, not the code to copy into this Python lesson.


**Manual experiment versus agent reference:** the executable cells below are yours to edit. The OpenClaw section at the end runs the fixed workshop recipe in `agent/lessons/`; it does not ingest your edited cells or reuse manual artifacts. Setup, displays, artifact bookkeeping, and terminal/process transport remain helpers.

### Start here
Run the executable cells in order. Edit the visible requests, rules, and experiment parameters to change your manual run. The artifact helper only copies prerequisites and records output paths. Starting again creates a fresh output folder.

In [ ]:
from pathlib import Path
import sys, json
from IPython.display import JSON, display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course/common.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course.agent.lesson_walkthrough import table, bars, loss_chart, show_skill_library
from course.agent.notebook_support import NotebookRun
from course.common import chat, request, rows, BASE_URL
import os, time

lesson = NotebookRun('07-specialist-tool')
OUTPUT = lesson.work
LIVE = True


### Configuration pattern · one route has one target

This excerpt is from our tested [Switchyard configuration](../run/workshop/configs/switchyard.toml). Read it alongside the [upstream quickstart](https://nvidia-nemo.github.io/Switchyard/getting_started/); it is not an adaptive classifier.

```toml
[routes.triage]
id = "workshop-triage"
type = "passthrough"
target = "specialist"
```

The wrapper sends `workshop-triage` to the router, then checks that the response identifies `specialist`. A mismatch is an error, not a silent fallback.


### 1 · Run both routes

Ask one general question and classify one recorded observation. These are fresh HTTP calls through Switchyard. Read the returned model IDs and timings.

In [ ]:
import json
from urllib.request import Request, urlopen
from switchyard_rust.server import Server
from course.common import BASE_URL
from run.workshop.workshop_agent.contracts import PERMITTED, ROUTING, RULES_PROMPT
def validate_item(item, source_count=1):
    errors = []
    if not isinstance(item, dict):
        return {"valid": False, "errors": ["Output is not a JSON object"], "destination": None}
    if set(item) != set(PERMITTED):
        errors.append("Expected exactly four specified fields")
    for field, values in PERMITTED.items():
        value = item.get(field)
        if not isinstance(value, str) or value not in values:
            errors.append(f"Invalid {field}")
    if item.get("confidence") == "CONFIRMED" and source_count < 2:
        errors.append("R1: one source cannot establish corroboration")
    sensor = item.get("sensor_quality")
    if isinstance(sensor, str) and sensor in {"DEGRADED", "UNKNOWN"} and item.get("confidence") != "POSSIBLE":
        errors.append("R2: degraded or unknown feed requires POSSIBLE")
    activity = item.get("activity_type")
    destination = ROUTING.get(activity) if isinstance(activity, str) else None
    return {"valid": not errors, "errors": errors,
            "destination": destination if not errors else None,
            "model_proposed": item.get("routing"),
            "routing_overridden": not errors and item.get("routing") != destination,
            "release": "HUMAN_REVIEW_REQUIRED"}

routes_toml = """schema_version = 1

[llm_clients.lightning]
format = "openai_chat"
base_url = "http://127.0.0.1:8002/v1"

[targets.base]
id = "base"
llm_client = "lightning"
extra_body = { chat_template_kwargs = { enable_thinking = false } }

[targets.specialist]
id = "specialist"
llm_client = "lightning"
extra_body = { chat_template_kwargs = { enable_thinking = false } }

# Explicit task routes are the measured baseline. No cross-contract fallback.
[routes.reason]
id = "workshop-reason"
type = "passthrough"
target = "base"

[routes.triage]
id = "workshop-triage"
type = "passthrough"
target = "specialist"
"""
routes_toml = routes_toml.replace('http://127.0.0.1:8002/v1', BASE_URL + '/v1')
routes_path = OUTPUT / 'routes.toml'
routes_path.write_text(routes_toml)
print(routes_toml)
routing_results = []
with Server(routes_path) as server:
    for route, expected, messages in [
        ('workshop-reason', 'base', [{'role': 'user', 'content': 'Why should a report cite its observation? Reply briefly.'}]),
        ('workshop-triage', 'specialist', [{'role': 'system', 'content': RULES_PROMPT},
            {'role': 'user', 'content': (ROOT / 'run/chain/descriptions/civil-market.txt').read_text()}]),
    ]:
        started = time.perf_counter()
        payload = {'model': route, 'messages': messages, 'temperature': 0, 'max_tokens': 512}
        if expected == 'specialist': payload['response_format'] = {'type': 'json_object'}
        http_request = Request(server.base_url + '/v1/chat/completions', data=json.dumps(payload).encode(),
                               headers={'Content-Type': 'application/json'})
        with urlopen(http_request, timeout=120) as response:
            reply = json.load(response)
        choice = reply['choices'][0]
        assert reply['model'] == expected and choice['finish_reason'] == 'stop'
        answer = choice['message']['content']
        routing_results.append({'route': route, 'model': reply['model'], 'finish_reason': choice['finish_reason'],
                                'seconds': time.perf_counter()-started, 'answer': answer})
        if expected == 'specialist':
            item = json.loads(answer)
            result = {'item': item, 'gate': validate_item(item, source_count=1)}
(OUTPUT / 'routing-results.json').write_text(json.dumps(routing_results, indent=2))


In [ ]:
table(["Route", "Actual model", "Finish", "Seconds", "Answer"], [(r["route"], r["model"], r["finish_reason"], r["seconds"], r["answer"]) for r in routing_results])

### 2 · Make the rules visible

| Proposal | Application behavior |
|---|---|
| Civil gathering with wrong destination | Route to CIMIC using the lookup table |
| One source claiming CONFIRMED | Reject |

These are deliberate test inputs. Passing a contract does not prove the scene was understood correctly.

In [ ]:
valid = {'activity_type':'civil_gathering','sensor_quality':'CLEAR',
         'confidence':'PROBABLE','routing':'J2-ANALYSIS'}
print('Routing overridden:', validate_item(valid,source_count=1))
invalid = {**valid,'confidence':'CONFIRMED'}
print('Rejected:', validate_item(invalid,source_count=1))
assert validate_item(valid,1)['destination'] == 'CIMIC'
assert not validate_item(invalid,1)['valid']

In [ ]:
check = validate_item
table(["Test", "Actual gate result"], [("Destination correction", check(valid,1)), ("Overconfidence rejection", check(invalid,1))])

### 3 · Try a tool too early

**No evidence → request draft → rejection**

The application enforces preconditions even if a model asks for the wrong tool. Run this intentional failure and inspect the rejection in the execution details.

In [ ]:
def allowed_actions(state, skill):
    if 'observation' not in state: return ['perceive', 'request_review']
    if 'triage' not in state: return ['triage', 'request_review']
    return ['draft', 'request_review'] if state['triage']['gate']['valid'] else ['request_review']

def require_draft_inputs(state):
    if 'draft' not in allowed_actions(state, 'draft-report'):
        raise ValueError('Draft requires evidence and a valid classification')
state = {'clip': 'civil-market', 'released': False}
try:
    require_draft_inputs(state)
except ValueError as exc:
    print('Expected rejection:', exc)
else:
    raise AssertionError('Premature drafting was accepted')


In [ ]:
print("Allowed at the start:", allowed_actions(state, "draft-report"))

### Check and save this manual run
Validate the artifacts and record this run for the next manual walkthrough. A successful check describes execution and evidence; read the actual quality scores separately.

In [ ]:
assert [r['model'] for r in routing_results] == ['base', 'specialist']
assert not validate_item(invalid, 1)['valid']
manual_result = lesson.finish({'routes': routing_results, 'invalid_confidence_rejected': True,
                               'premature_draft_rejected': True})
display(JSON(manual_result, expanded=False))

### Grow the skill library
The library now includes this lesson’s skill, **workshop-specialist-tool**. Open the actual instructions below.

**`inspect-tool` → `test-tool` → `check-tool`**

In [ ]:
show_skill_library('07-specialist-tool')

### Prepare OpenClaw
Run once for each new demo. This cell installs the growing skill library in a fresh sandbox workspace and starts a background host worker for this lesson’s three actions. **It does not ask the agent to do anything yet.**

Here, **installing a skill means placing its `SKILL.md` instruction file in OpenClaw’s workspace**. The current lesson’s source is `course/agent/skills/workshop-specialist-tool/SKILL.md` in this repository.

Each run gets this structure inside the OpenShell sandbox (launcher files omitted):

```text
/home/sandbox/course-runs/openclaw-<run-id>/
├── openclaw.json                 # points OpenClaw at this run’s workspace
├── chapter_tool.py              # helper invoked by the agent
├── chapter.json                 # this lesson’s task and allowed actions
└── workspace/
    ├── AGENTS.md                 # workspace instructions
    └── skills/
        ├── workshop-specialist-tool/
        │   └── SKILL.md
        └── ...                  # skills from earlier notebooks
```

OpenClaw discovers skills under `workspace/skills/`. The task you paste in the next step tells the agent which skill to read and supplies the full path to `chapter_tool.py` (the skill’s `<tool_path>` placeholder).

The agent invokes the helper through its execution tool. The helper forwards the selected action to the host worker, which runs the fixed lesson code and returns the result. Nano follows the skill and coordinates the calls.

This lesson does not require artifacts from an earlier OpenClaw demo.

Start the TUI task within **15 minutes** and keep this run’s `terminal_run` value for collecting results. If the worker expires, prepare a new run. The local run directory stores host results; the installed skills live at the sandbox paths printed below.


In [ ]:
from course.agent.chapter_terminal import prepare_chapter_terminal, collect_chapter_terminal
from course.agent.notebook_terminal import open_terminal
terminal_run = prepare_chapter_terminal('07-specialist-tool')
print("Local run directory:", terminal_run["directory"])
print("OpenClaw sandbox workspace:", terminal_run["remote"] + "/workspace")
print("Installed skill:", terminal_run["remote"] + "/workspace/skills/workshop-specialist-tool/SKILL.md")
print("Python helper:", terminal_run["remote"] + "/chapter_tool.py")
print("\nThe agent will run these commands individually:")
for action in ['inspect-tool', 'test-tool', 'check-tool']:
    print("python3 " + terminal_run["remote"] + "/chapter_tool.py " + action)

### Open the TUI and paste the task
Exit the previous TUI with `/exit` first. This cell opens a terminal on the workshop computer’s desktop. Copy only the generated task into OpenClaw.

The terminal label is **Nano**, its lightweight orchestration model. Nano orchestrates the lesson tools. Switchyard routes their requests to Lightning base or specialist. Application code validates the returned classification.

In [ ]:
open_terminal(terminal_run)
print("\nPASTE THIS INTO OPENCLAW:\n")
print(terminal_run["prompt"])

### Bring the evidence back
After the TUI finishes, run this cell. It retrieves that session’s trace and checks the artifacts and skill reads. If Run All reaches this cell before you finish the TUI task, run it again afterward.

**Look for:** General → base; classification → specialist; actual model IDs checked.

In [ ]:
agent_result = collect_chapter_terminal(terminal_run)
display(JSON(agent_result, expanded=True))

**What you demonstrated:** The model proposes an answer or action. Code enforces the contract and order.

Exit the TUI with `/exit` when done.